<a href="https://colab.research.google.com/github/springboardmentor2501/Shipment-Sure/blob/Almeehra-Shaikh/milestone1_anomalydataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import openpyxl

In [5]:
import pandas as pd
import numpy as np

# Load original dataset
df = pd.read_csv("/content/smart_logistics_dataset.csv")
df.head()

,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,2024-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38,320,4,NaN,60.1,285,1
1,2024-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,16,439,7,Weather,80.9,174,1
2,2024-07-29 18:42:48,Truck_10,54.9232,79.5455,190,In Transit,25.2,62.2,Detour,34,355,3,NaN,99.2,260,0
3,2024-10-28 00:50:54,Truck_9,42.3900,-1.4788,330,Delivered,25.4,52.3,Heavy,37,227,5,Traffic,97.4,160,1
4,2024-09-27 15:52:58,Truck_7,-65.8477,47.9468,480,Delayed,20.5,57.2,Clear,56,197,6,NaN,71.6,270,1


In [6]:

# Create a copy of the original dataset
df_anom = df.copy()

# 1. Introduce Null values
for col in df_anom.columns[:3]:
    df_anom.loc[df_anom.sample(frac=0.01).index, col] = np.nan

# 2. Duplicate rows
duplicate_rows = df_anom.sample(50)
df_anom = pd.concat([df_anom, duplicate_rows], ignore_index=True)

# 3. Datatype mismatch (convert some numeric values to strings)
numeric_cols = df_anom.select_dtypes(include=[np.number]).columns[:2]

for col in numeric_cols:
    idx = df_anom.sample(30).index

    # Convert whole column to object before inserting strings
    df_anom[col] = df_anom[col].astype(object)

    df_anom.loc[idx, col] = df_anom.loc[idx, col].astype(str)


# 4. Outliers: Inflate values
if len(numeric_cols) > 0:
    outlier_idx = df_anom.sample(20).index
    df_anom.loc[outlier_idx, numeric_cols[0]] = pd.to_numeric(
        df_anom.loc[outlier_idx, numeric_cols[0]], errors='coerce'
    ) * 50

# 5. Logical errors
if "region" in df_anom.columns:
    df_anom.loc[df_anom.sample(30).index, "region"] = "Nest"

# Shipment date before order date
date_cols = [c for c in df_anom.columns if "date" in c.lower()]
if len(date_cols) >= 2:
    order_col, ship_col = date_cols[:2]
    idx = df_anom.sample(20).index
    df_anom.loc[idx, ship_col] = pd.to_datetime(df_anom.loc[idx, order_col]) - pd.Timedelta(days=3)

df_anom.head()

,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,2024-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38,320,4,NaN,60.1,285,1
1,2024-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,16,439,7,Weather,80.9,174,1
2,2024-07-29 18:42:48,Truck_10,54.9232,79.5455,190,In Transit,25.2,62.2,Detour,34,355,3,NaN,99.2,260,0
3,2024-10-28 00:50:54,Truck_9,42.39,-1.4788,330,Delivered,25.4,52.3,Heavy,37,227,5,Traffic,97.4,160,1
4,2024-09-27 15:52:58,Truck_7,-65.8477,47.9468,480,Delayed,20.5,57.2,Clear,56,197,6,NaN,71.6,270,1


In [7]:
# Save as CSV with UTF-8 encoding
df_anom.to_csv("shipment_dataset_anomalous.csv", index=False, encoding='utf-8-sig')
print("Anomalous file saved as shipment_dataset_anomalous.csv")

Anomalous file saved as shipment_dataset_anomalous.csv


In [10]:
import pandas as pd
import numpy as np

# Load the anomalous dataset (CSV version)
df_clean = pd.read_csv("/content/shipment_dataset_anomalous.csv", encoding="utf-8-sig")

print("Before cleaning:")
print(df_clean.info())
print(df_clean.head())

# 1. FIX NULL / MISSING VALUES

# Option 1: Forward fill, then backward fill
df_clean = df_clean.ffill().bfill()

# 2. REMOVE DUPLICATES

df_clean.drop_duplicates(inplace=True)


# 3. FIX DATATYPE MISMATCHES
#    (Some numeric columns contain strings because of anomaly)


for col in df_clean.columns:
    try:
        df_clean[col] = pd.to_numeric(df_clean[col])
    except:
        pass

# Re-run conversion for columns that still have object type but should be numeric
force_numeric_cols = []
for col in df_clean.columns:
    # Detect numeric-looking strings (e.g. "1234", "55.6")
    if df_clean[col].dtype == "object":
        if df_clean[col].astype(str).str.replace('.', '', 1).str.isdigit().any():
            force_numeric_cols.append(col)

for col in force_numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")


# 4. HANDLE OUTLIERS USING IQR CAPPING


numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap values outside the range
    df_clean[col] = df_clean[col].clip(lower, upper)

# 5. FIX LOGICAL ERRORS - INVALID REGION NAMES

if "region" in df_clean.columns:
    # Assume the most frequent region is correct
    correct_region = df_clean["region"].mode()[0]

    # Replace invalid regions such as "Nest"
    df_clean.loc[~df_clean["region"].isin([correct_region]), "region"] = correct_region


# 6. FIX LOGICAL ERRORS - SHIPMENT DATE < ORDER DATE

# Find all date columns
date_cols = [col for col in df_clean.columns if "date" in col.lower()]

# Ensure date columns are proper datetime
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

# Fix date order problem
if len(date_cols) >= 2:
    order_col, ship_col = date_cols[:2]   # first 2 date columns
    df_clean.loc[df_clean[ship_col] < df_clean[order_col], ship_col] = df_clean[order_col]


# SAVE CLEANED DATA

df_clean.to_csv("shipment_dataset_cleaned.csv", index=False, encoding="utf-8-sig")

print("\nCleaning complete! Saved as shipment_dataset_cleaned.csv")
print(df_clean.info())
df_clean.head()

Before cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Timestamp                1040 non-null   object 
 1   Asset_ID                 1040 non-null   object 
 2   Latitude                 1040 non-null   float64
 3   Longitude                1050 non-null   float64
 4   Inventory_Level          1050 non-null   int64  
 5   Shipment_Status          1050 non-null   object 
 6   Temperature              1050 non-null   float64
 7   Humidity                 1050 non-null   float64
 8   Traffic_Status           1050 non-null   object 
 9   Waiting_Time             1050 non-null   int64  
 10  User_Transaction_Amount  1050 non-null   int64  
 11  User_Purchase_Frequency  1050 non-null   int64  
 12  Logistics_Delay_Reason   772 non-null    object 
 13  Asset_Utilization        1050 non-null   float64
 14  Demand_

,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,User_Transaction_Amount,User_Purchase_Frequency,Logistics_Delay_Reason,Asset_Utilization,Demand_Forecast,Logistics_Delay
0,2024-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38,320,4,Weather,60.1,285,1
1,2024-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,16,439,7,Weather,80.9,174,1
2,2024-07-29 18:42:48,Truck_10,54.9232,79.5455,190,In Transit,25.2,62.2,Detour,34,355,3,Weather,99.2,260,0
3,2024-10-28 00:50:54,Truck_9,42.3900,-1.4788,330,Delivered,25.4,52.3,Heavy,37,227,5,Traffic,97.4,160,1
4,2024-09-27 15:52:58,Truck_7,-65.8477,47.9468,480,Delayed,20.5,57.2,Clear,56,197,6,Traffic,71.6,270,1
